# Boomerang Sampler Benchmarks — Synthetic Logistic Regression

Two experiments inspired by the PDMP literature:

| Experiment | Target | Samplers | What it tests |
|---|---|---|---|
| **A. Dense** | `logreg_synthetic(n=500, p=10, dense)` | Boomerang PLI, Sticky PLI | Continuous posterior — sticky should recover the Boomerang |
| **B. Sparse** | `logreg_synthetic(n=500, p=10, sparse)` | Boomerang PLI, Sticky PLI | True zeros — sticky should find them, Boomerang cannot |

Both use Gaussian prior (scale=1.0) and PLI thinning throughout. Single run, N=10000 skeleton points.

In [ ]:
import os, sys
os.chdir('..')
import numpy as np
from benchmarks_august.targets.logreg import logreg_synthetic
from benchmarks_august.samplers import build_sampler, build_kappa, apply_preprocess
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path, resample_sticky_pdmp_path
from sampler_eval import sample_quality, model_performance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [ ]:
# Shared settings
N = 10000
n_resample = 50000
burnin = 0.2
refresh_rate = 1.0

---
## Experiment A — Dense target (p=10, all coefficients nonzero)

In [ ]:
target_dense = logreg_synthetic(n=500, p=10, sparsity='dense', seed=42,
                                prior={'kind': 'gaussian', 'scale': 1.0})
print('beta_true:', target_dense.true_params)

X_d, y_d = target_dense.data['X'], target_dense.data['y']
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_d, y_d, test_size=0.2, random_state=0)

# Rebuild target on train split for fair evaluation
from benchmarks_august.targets.logreg import logreg
target_dense_train = logreg(X_tr_d, y_tr_d, prior={'kind': 'gaussian', 'scale': 1.0})

# sklearn reference
lr_d = LogisticRegression(C=1.0, penalty='l2', fit_intercept=False, max_iter=1000).fit(X_tr_d, y_tr_d)
print('sklearn MAP:', lr_d.coef_[0])

In [ ]:
# --- Boomerang PLI (dense) ---
boom_d = build_sampler('boomerang_pli', target_dense_train, N=N, refresh_rate=refresh_rate)
apply_preprocess(boom_d, target_dense_train, {'method': 'diagonal'})
boom_d.sample_auto(diagnostics=True)

In [ ]:
# --- Sticky Boomerang PLI (dense) ---
kappa_d = build_kappa({'kind': 'uniform', 'gamma_prior': 0.5}, target_dense_train)
sticky_d = build_sampler('sticky_boomerang_pli', target_dense_train, N=N,
                         kappa=kappa_d, refresh_rate=refresh_rate)
apply_preprocess(sticky_d, target_dense_train, {'method': 'diagonal'})
sticky_d.sample_auto(diagnostics=True)

In [ ]:
# Resample & visualize
_, x_boom_d = resample_pdmp_path(boom_d, n_samples=n_resample, burnin_frac=burnin)
_, x_sticky_d = resample_sticky_pdmp_path(sticky_d, n_samples=n_resample, burnin_frac=burnin)

fig1 = sample_quality(x_boom_d, sklearn_coefs=lr_d.coef_[0], label='Boomerang PLI (dense)', burnin_frac=0)
fig2 = sample_quality(x_sticky_d, sklearn_coefs=lr_d.coef_[0], label='Sticky PLI (dense)', burnin_frac=0)

In [ ]:
# Performance
model_performance(X_tr_d, y_tr_d, X_te_d, y_te_d, {
    'Boomerang PLI': x_boom_d,
    'Sticky PLI': x_sticky_d,
}, burnin_frac=0)  # already burned in

---
## Experiment B — Sparse target (p=10, only 3 nonzero)

In [ ]:
target_sparse = logreg_synthetic(n=500, p=10, sparsity='sparse', seed=42,
                                 prior={'kind': 'gaussian', 'scale': 1.0})
print('beta_true:', target_sparse.true_params)

X_s, y_s = target_sparse.data['X'], target_sparse.data['y']
X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(X_s, y_s, test_size=0.2, random_state=0)

target_sparse_train = logreg(X_tr_s, y_tr_s, prior={'kind': 'gaussian', 'scale': 1.0})

lr_s = LogisticRegression(C=1.0, penalty='l2', fit_intercept=False, max_iter=1000).fit(X_tr_s, y_tr_s)
print('sklearn MAP:', lr_s.coef_[0])

In [ ]:
# --- Boomerang PLI (sparse) ---
boom_s = build_sampler('boomerang_pli', target_sparse_train, N=N, refresh_rate=refresh_rate)
apply_preprocess(boom_s, target_sparse_train, {'method': 'diagonal'})
boom_s.sample_auto(diagnostics=True)

In [ ]:
# --- Sticky Boomerang PLI (sparse) ---
kappa_s = build_kappa({'kind': 'uniform', 'gamma_prior': 0.5}, target_sparse_train)
sticky_s = build_sampler('sticky_boomerang_pli', target_sparse_train, N=N,
                         kappa=kappa_s, refresh_rate=refresh_rate)
apply_preprocess(sticky_s, target_sparse_train, {'method': 'diagonal'})
sticky_s.sample_auto(diagnostics=True)

In [ ]:
_, x_boom_s = resample_pdmp_path(boom_s, n_samples=n_resample, burnin_frac=burnin)
_, x_sticky_s = resample_sticky_pdmp_path(sticky_s, n_samples=n_resample, burnin_frac=burnin)

fig3 = sample_quality(x_boom_s, sklearn_coefs=lr_s.coef_[0], label='Boomerang PLI (sparse)', burnin_frac=0)
fig4 = sample_quality(x_sticky_s, sklearn_coefs=lr_s.coef_[0], label='Sticky PLI (sparse)', burnin_frac=0)

In [ ]:
model_performance(X_tr_s, y_tr_s, X_te_s, y_te_s, {
    'Boomerang PLI': x_boom_s,
    'Sticky PLI': x_sticky_s,
}, burnin_frac=0)

---
## Sparsity comparison (Experiment B)
The key metric for the sparse target: what fraction of time does each coordinate spend at exactly zero?

In [ ]:
import matplotlib.pyplot as plt

D = target_sparse.D
zero_frac_sticky = (x_sticky_s == 0).mean(axis=0)
true_zero = (target_sparse.true_params == 0).astype(float)

fig, ax = plt.subplots(figsize=(10, 4))
coords = np.arange(D)
ax.bar(coords - 0.15, true_zero, width=0.3, label='truly zero', color='black', alpha=0.3)
ax.bar(coords + 0.15, zero_frac_sticky, width=0.3, label='sticky zero frac', color='steelblue')
ax.set_xticks(coords)
ax.set_xticklabels([f'β{i}' for i in range(D)])
ax.set_ylabel('fraction at zero')
ax.set_title('Variable selection: Sticky PLI vs ground truth')
ax.legend()
plt.tight_layout()